# tinyLMTune — Named Entity Recognition (NER)

This notebook demonstrates 3 ways to train TinyBERT for **ner** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (CoNLL-2003)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [1]:
# import logging
# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# # Install if needed (uncomment):
!pip install -e ../../tinylmtune_v2/



Obtaining file:///home/sagemaker-user/SLM/tinylmtune_v2
  Preparing metadata (setup.py) ... done
  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached multiprocess-0.70.16-py311-none-any.whl (143 kB)
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
  Attempting uninstall: datasets
    Found existing installation: datasets 2.2.1
    Uninstalling datasets-2.2.1:
      Successfully uninstalled datasets-2.2.1
  DEPRECATION: Legacy editable install of tinylmtune==

ModuleNotFoundError: No module named 'tinylmtune'

In [1]:
from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation

2026-05-19 06:38:52.458122: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-19 06:38:52.470243: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-19 06:38:52.474085: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-19 06:38:52.483969: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [ ]:
best = optimize_slm(
    task="ner",
    corpus_prompt="Generate sentences with named entities about business, politics, and sports",
    n_examples=5000,
    pop_size=4,
    generations=2,
    output_dir="models/ner_synthetic",
)
print("Best config:", best)

### Inference on synthetic model

In [3]:
model = TinyInference("models/ner_synthetic")
result = model.predict("Elon Musk founded SpaceX in Los Angeles.")
print(result)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'text': 'Elon Musk founded SpaceX in Los Angeles.', 'entities': []}


---
## Example 2 — Benchmark Data (CoNLL-2003)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load CoNLL-2003 dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("eriktks/conll2003", split="train")
ds = ds.shuffle(seed=42).select(range(1000))

tag_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]

benchmark_data = []
for r in ds:
    tokens = r["tokens"]
    ner_tags = r["ner_tags"]
    text = " ".join(tokens)

    entities, current = [], None
    char_pos = 0
    for tok, tag_id in zip(tokens, ner_tags):
        tag = tag_names[tag_id] if tag_id < len(tag_names) else "O"
        tok_start = text.find(tok, char_pos)
        tok_end = tok_start + len(tok)
        if tag.startswith("B-"):
            if current: entities.append(current)
            current = {"text": tok, "label": tag[2:], "start": tok_start, "end": tok_end}
        elif tag.startswith("I-") and current and tag[2:] == current["label"]:
            current["text"] = text[current["start"]:tok_end]
            current["end"] = tok_end
        else:
            if current: entities.append(current); current = None
        char_pos = tok_end
    if current: entities.append(current)
    benchmark_data.append({"text": text, "entities": entities})

print(f"Loaded {len(benchmark_data)} records")
print(f"Sample: {benchmark_data[0]}")

### Analyze token lengths

In [ ]:
print_token_analysis(benchmark_data, task="ner")

### Check recommended search space

In [ ]:
print_recommendation(n_samples=len(benchmark_data), task="ner")

### Train with GA optimisation

In [ ]:
best = optimize_slm(
    task="ner",
    user_data=benchmark_data,
    pop_size=4,
    generations=2,
    max_len=32,
    output_dir="models/ner_benchmark",
)
print("Best config:", best)

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [ ]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

### Inference on benchmark model

In [ ]:
model = TinyInference("models/ner_benchmark")
result = model.predict("Tim Cook announced new products at Apple headquarters in Cupertino.")
print(result)

---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [ ]:
my_data = [
    {"text": "John Smith works at Google in California",
     "entities": [{"text": "John Smith", "label": "PER", "start": 0, "end": 10},
                  {"text": "Google", "label": "ORG", "start": 20, "end": 26},
                  {"text": "California", "label": "LOC", "start": 30, "end": 40}]},
    {"text": "Apple released the iPhone in Cupertino",
     "entities": [{"text": "Apple", "label": "ORG", "start": 0, "end": 5},
                  {"text": "iPhone", "label": "MISC", "start": 18, "end": 24},
                  {"text": "Cupertino", "label": "LOC", "start": 28, "end": 37}]},
    {"text": "Barack Obama visited the United Nations in New York",
     "entities": [{"text": "Barack Obama", "label": "PER", "start": 0, "end": 12},
                  {"text": "United Nations", "label": "ORG", "start": 25, "end": 39},
                  {"text": "New York", "label": "LOC", "start": 43, "end": 51}]},
    {"text": "Tesla announced record sales in Shanghai last quarter",
     "entities": [{"text": "Tesla", "label": "ORG", "start": 0, "end": 5},
                  {"text": "Shanghai", "label": "LOC", "start": 31, "end": 39}]},
    {"text": "The weather is nice today",
     "entities": []},
    {"text": "Microsoft acquired GitHub for 7.5 billion dollars",
     "entities": [{"text": "Microsoft", "label": "ORG", "start": 0, "end": 9},
                  {"text": "GitHub", "label": "ORG", "start": 19, "end": 25}]},
    {"text": "Angela Merkel served as Chancellor of Germany for 16 years",
     "entities": [{"text": "Angela Merkel", "label": "PER", "start": 0, "end": 13},
                  {"text": "Germany", "label": "LOC", "start": 38, "end": 45}]},
    {"text": "The Olympic Games were held in Tokyo in 2021",
     "entities": [{"text": "Olympic Games", "label": "MISC", "start": 4, "end": 17},
                  {"text": "Tokyo", "label": "LOC", "start": 31, "end": 36}]},
    {"text": "It was a normal day with nothing remarkable happening",
     "entities": []},
    {"text": "Jeff Bezos founded Amazon in his garage in Seattle",
     "entities": [{"text": "Jeff Bezos", "label": "PER", "start": 0, "end": 10},
                  {"text": "Amazon", "label": "ORG", "start": 19, "end": 25},
                  {"text": "Seattle", "label": "LOC", "start": 43, "end": 50}]},
]

best = optimize_slm(
    task="ner",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/ner_user",
)

### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw sentences — Flan-T5 extracts entities automatically
raw_texts = [
    "Sundar Pichai announced Google's new AI features at their Mountain View campus.",
    "The European Union imposed sanctions on Russian energy imports.",
    "Liverpool defeated Manchester United 3-1 at Anfield stadium.",
    "NASA's Perseverance rover collected samples on Mars.",
    "Warren Buffett's Berkshire Hathaway reported strong quarterly results in Omaha.",
]

best = optimize_slm(
    task="ner",
    user_data=raw_texts,
    pop_size=4,
    generations=1,
    output_dir="models/ner_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts without entity annotations
wrong_format = [
    {"sentence": "Mark Zuckerberg leads Meta from their headquarters in Menlo Park."},
    {"sentence": "The United Nations held its climate summit in Glasgow last November."},
    {"sentence": "Cristiano Ronaldo scored twice for Al Nassr in Riyadh."},
]

best = optimize_slm(
    task="ner",
    user_data=wrong_format,
    pop_size=4,
    generations=1,
    output_dir="models/ner_wrong",
)

### Visualize user data results

In [ ]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

### Inference

In [ ]:
model = TinyInference("models/ner_user")
result = model.predict("Satya Nadella leads Microsoft from their Redmond headquarters.")
print(result)

---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | CoNLL-2003 | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).